In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "pdf.fonttype": 42,
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath, amssymb}",
    "font.family": "serif",
    "font.size": 16,
    "axes.labelsize": 20,
    "axes.titlesize": 18,
    "legend.fontsize": 14,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
})

data_file = Path("../Sim_Data/Cosmos/V_Cosmo.npz")
output_dir = Path("Plots/NSC_pert/ALP_perturbations/ALP_delta_a")
output_dir.mkdir(parents=True, exist_ok=True)

ymin = 1e-5
ymax = 1e1
eps = 1e-50
kappa_nl_analytic = 285.0
kappa_J_b0_override = 2.0 * 123.33835713588856

lab_k = r"$\kappa$"
lab_da10 = r"$|\delta_a(\kappa,x=10)|$"

color_sc_b4 = "#31688e"
color_nsc_b4 = "#440154"
color_ratio_b4 = "#8e0d0d"
color_sc_b0 = "#0dca00"
color_nsc_b0 = "#b12a90"
color_ratio_b0 = "#003837"

with np.load(data_file, allow_pickle=True) as data:
    b_values = data["b_values"]
    cosmo_names = data["cosmo_names"]
    kappas = data["kappas"]
    delta_a = data["delta_a_abs"]
    kappa_J = data["kappa_J"]
    kappa_osc = data["kappa_osc"]
    x_spec = data["x_spec"]
    inputs = data["inputs"].item()

b_idx = {int(b): i for i, b in enumerate(b_values)}
cosmo_idx = {name: i for i, name in enumerate(cosmo_names)}

def get_run(b, cosmo):
    i = b_idx[b]
    j = cosmo_idx[cosmo]
    return {
        "kappas": kappas,
        "delta_a": delta_a[i, j],
        "kappa_J": float(kappa_J[i, j]),
        "kappa_osc": float(kappa_osc[i, j]),
        "x_spec": float(x_spec[i, j]),
    }

b0_sc = get_run(0, "SC")
b0_nsc = get_run(0, "NSC")
b4_sc = get_run(4, "SC")
b4_nsc = get_run(4, "NSC")

b0_nsc["kappa_J"] = kappa_J_b0_override

def add_nl_band(ax, kmax):
    ax.axvspan(kappa_nl_analytic, kmax, color="gray", alpha=0.22, edgecolor="none", zorder=0, label=r"$\kappa \geq \kappa_{{\rm nl},\phi}$")

def plot_panel_separate(ax, run_nsc, label_nsc, color_nsc, run_sc, label_sc, color_sc):
    k = run_nsc["kappas"]
    kmax = k[-1]

    add_nl_band(ax, kmax)
    ax.plot(k, run_nsc["delta_a"], color=color_nsc, label=label_nsc, zorder=3)
    ax.plot(k, run_sc["delta_a"], color=color_sc, label=label_sc, zorder=3)
    ax.axvline(run_nsc["kappa_osc"], color="gray", linestyle=":", alpha=0.7, zorder=2, label=r"$\kappa_{\rm osc,NSC}$")
    ax.axvline(run_nsc["kappa_J"], color=color_nsc, linestyle="--", alpha=0.7, zorder=2)
    ax.plot([], [], color="black", linestyle="--", alpha=0.7, label=r"$\kappa_{J,\rm NSC}$")
    ax.axhline(1.0, color="black", linestyle="--", alpha=0.7, zorder=1)
    ax.set_xlim(k[0], kmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylabel(lab_da10)
    ax.set_xlabel(lab_k)
    ax.legend(loc="upper left", prop={"size": 12}, framealpha=1)

def plot_panel_ratio(ax, run_nsc, run_sc, color, ylabel, kJ_label):
    k = run_nsc["kappas"]
    ratio = run_nsc["delta_a"] / (run_sc["delta_a"] + eps)

    add_nl_band(ax, k[-1])
    ax.plot(k, ratio, color=color, label=rf"$x={run_nsc['x_spec']:.2f}$", zorder=3)
    ax.axvline(run_nsc["kappa_J"], color=color, linestyle="--", alpha=0.7, zorder=2, label=kJ_label)
    ax.axvline(run_nsc["kappa_osc"], color="gray", linestyle=":", alpha=0.7, zorder=2, label=r"$\kappa_{\rm osc,NSC}$")
    ax.axhline(1.0, color="black", linestyle="--", alpha=0.7, zorder=1)
    ax.set_xlim(k[0], k[-1])
    ax.set_ylim(1e-1, 4e4)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylabel(ylabel)
    ax.set_xlabel(lab_k)
    ax.legend(loc="upper left", prop={"size": 12}, framealpha=1)

fig, axs = plt.subplots(2, 2, figsize=(13, 9))

plot_panel_separate(axs[0, 0], b4_nsc, r"$b=4,\mathrm{NSC}$", color_nsc_b4, b4_sc, r"$b=4,\mathrm{SC}$", color_sc_b4)
plot_panel_ratio(axs[0, 1], b4_nsc, b4_sc, color_ratio_b4, r"$|\delta_{a,\mathrm{NSC}}^{(b=4)}|/|\delta_{a,\mathrm{SC}}^{(b=4)}|$", r"$\kappa_{J,\mathrm{NSC}}^{(b=4)}$")
plot_panel_separate(axs[1, 0], b0_nsc, r"$b=0,\mathrm{NSC}$", color_nsc_b0, b0_sc, r"$b=0,\mathrm{SC}$", color_sc_b0)
plot_panel_ratio(axs[1, 1], b0_nsc, b0_sc, color_ratio_b0, r"$|\delta_{a,\mathrm{NSC}}^{(b=0)}|/|\delta_{a,\mathrm{SC}}^{(b=0)}|$", r"$\kappa_{J,\mathrm{NSC}}^{(b=0)}$")

fig.tight_layout()
fig.savefig(output_dir / "Spectra_cosmosb_1.pdf", bbox_inches="tight", dpi=300)
plt.show()
